# Board evaluation

In [46]:
import torch
import pandas as pd
import os
import chess
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

DATASET_DIR = os.path.join("..", "dataset")
MAIN_DATASET = os.path.join(DATASET_DIR, "chessData.csv")
df = pd.read_csv(MAIN_DATASET)

df["Evaluation"] = pd.to_numeric(df["Evaluation"], downcast="integer", errors="coerce")

MODEL_PATH = os.path.join("..", "catfish_eval.pt")

In [47]:
class ChessDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fen = self.df.iloc[idx]["FEN"]
        eval_score = self.df.iloc[idx]["Evaluation"]
        board = fen_to_tensor(fen)
        target = torch.tensor([eval_score], dtype=torch.float32)
        return board, target

dataset = ChessDataset(df)
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)

In [48]:
def get_en_passant(fen):
    matrix = torch.zeros((8,8), dtype=torch.float32)
    fen_en_passant = fen.split(" ")[3]
    if fen_en_passant != "-":
        square = chess.parse_square(fen_en_passant)
        matrix[square % 8][square % 8] = 1
    return matrix

def get_piece_matrix(board, piece_type, color):
    matrix = torch.zeros((8,8), dtype=torch.float32)
    for sq in board.pieces(piece_type, color):
        r = 7 - chess.square_rank(sq)
        c = chess.square_file(sq)
        matrix[r, c] = 1
    return matrix

def fen_to_tensor(fen):
    board = chess.Board(fen)
    piece_index = {
        chess.PAWN : 1,
        chess.ROOK : 2,
        chess.KNIGHT : 3,
        chess.BISHOP : 4,
        chess.QUEEN : 5,
        chess.KING : 6,
    }
    fen_tensor = torch.zeros((13,8,8), dtype=torch.float32)
    fen_tensor[0] = get_en_passant(fen)
    for piece, val in piece_index.items():
        fen_tensor[val] = get_piece_matrix(board, piece, chess.BLACK)
        fen_tensor[val + 6] = get_piece_matrix(board, piece, chess.WHITE)
    return fen_tensor

In [49]:
class Catfish(nn.Module):
    def __init__(self):
        super(Catfish, self).__init__()
        self.conv1 = nn.Conv2d(13, 26, 3, padding=1)
        self.conv2 = nn.Conv2d(26, 52, 3, padding=1)
        self.fc1 = nn.Linear(52 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
model = Catfish()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    for boards, targets in loader:
        predictions = model(boards)
        loss = criterion(predictions, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print("epoch:", epoch, "loss:", loss.item())
    
torch.save(model.state_dict(), MODEL_PATH)

In [ ]:
def evaluate_board(fen):
    model.load_state_dict(torch.load(MODEL_PATH))
    model.eval()
    board = fen_to_tensor(fen)
    board = board.unsqueeze(0)
    with torch.no_grad():
        evaluation = model(board)
    return evaluation.item()

evaluate_board("r1bqkb1r/pp1n1ppp/2n1p3/2ppP3/3P1P2/2P5/PP1N2PP/R1BQKBNR w KQkq - 1 7") #+86

36.95035934448242